# Hava durumu entegrasyonu

Modelin en büyük açığı mevsimsellikti: eğitim ayları ile test ayları arasında `yilin_gunu`
açısından hiç örtüşme yok, yani model yaz seviyelerini takvimden okuyamıyor. Sıcaklık bu
sorunu çözüyor — 30 derece, hangi yılın hangi ayı olduğundan bağımsız olarak 30 derecedir.

Veri kaynağı: Open-Meteo arşiv API'si (ücretsiz, anahtar gerektirmiyor). Test dönemi
(Nisan-Temmuz 2026) geçmişte kaldığı için **tahmin edilmiş değil, gerçekleşmiş** sıcaklıkları
kullanabiliyoruz. Tahmin anında bilinen bir değişken olduğu için de bayatlamıyor:
122 günlük ufkun ilk günü de son günü de aynı tazelikte.

Çekim kodu 03_deneyler.ipynb'de, sonuç `data/external/hava_durumu.csv` dosyasında.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RAW, EXT = '../data/raw/', '../data/external/'
train = pd.read_csv(RAW + 'train.csv')
test  = pd.read_csv(RAW + 'test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])

train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    med = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = med

TATILLER = pd.to_datetime([
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23','2025-05-01',
    '2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09','2025-08-30','2025-10-29',
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-03-22','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30']).date

def temel(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il'], df['bolge'] = lok[0], lok[1]
    df['ilce'] = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')
    df['ay'] = df['tarih'].dt.month
    df['gun'] = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu'] = (df['haftanin_gunu'] >= 5).astype(int)
    df['yilin_gunu'] = df['tarih'].dt.dayofyear
    df['tatil'] = df['tarih'].dt.date.isin(TATILLER).astype(int)
    df['guc_bucket'] = pd.cut(df['guc'], bins=[0,100,250,500,1000,2000,5000,np.inf])
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    # Manisa'da ilçe kırılımı yok, orada bölge adı hava durumu anahtarı oluyor
    df['yer'] = np.where(df['ilce'] != 'YOK', df['ilce'], df['bolge'])
    return df

train, test = temel(train), temel(test)
train['log_t'] = np.log1p(train['tuketim_duzeltilmis'])
KATEGORIK = ['il','bolge','ilce','guc_bucket_str']
print(train.shape, test.shape)

## Sıcaklık özellikleri

Üç fikir üzerine kurulu:

**Derece-gün.** Tüketim sıcaklıkla düz değil, V şeklinde ilişkili — hem sıcakta (klima)
hem soğukta (ısıtma) artıyor, ılıman havada dibe iniyor. Tek bir sıcaklık sütunu bu V'yi
zor öğrenir; CDD (soğutma) ve HDD (ısıtma) olarak ayırınca her iki taraf da doğrusallaşıyor.
Eşik değerleri denendi, 22°C/18°C en iyi sonucu verdi.

**Termal atalet.** Binalar bir günde ısınıp soğumaz. Üç gündür sıcaksa klimalar sürekli
çalışıyordur; tek günlük sıcak dalgada o etki henüz oluşmamıştır. 3/7/14 günlük hareketli
ortalamalar bu gecikmeyi yakalıyor ve tek başına en büyük katkıyı veren grup bu oldu.

**Max/min ayrımı.** Kıyı ilçeleri ortalamada sıcak görünüyor (deniz geceleri ılıman tutuyor),
iç kesimler gündüz daha sıcak. Ortalama tek başına bu farkı gizliyor.

In [ ]:
hava = pd.read_csv(EXT + 'hava_durumu.csv', parse_dates=['tarih'])
hava = hava.sort_values(['yer','tarih']).reset_index(drop=True)

hava['cdd22'] = (hava['sic_ort'] - 22).clip(lower=0)
hava['hdd18'] = (18 - hava['sic_ort']).clip(lower=0)
hava['gunluk_fark'] = hava['sic_max'] - hava['sic_min']
hava['cdd_max30'] = (hava['sic_max'] - 30).clip(lower=0)

for p in [3, 7, 14]:
    hava[f'cdd22_{p}gun'] = hava.groupby('yer')['cdd22'].transform(lambda s: s.rolling(p, min_periods=1).mean())
for p in [3, 7]:
    hava[f'sic_{p}gun'] = hava.groupby('yer')['sic_ort'].transform(lambda s: s.rolling(p, min_periods=1).mean())
    hava[f'hdd18_{p}gun'] = hava.groupby('yer')['hdd18'].transform(lambda s: s.rolling(p, min_periods=1).mean())
hava['sic_max_3gun'] = hava.groupby('yer')['sic_max'].transform(lambda s: s.rolling(3, min_periods=1).mean())

HAVA = ['sic_ort','sic_max','sic_min','cdd22','hdd18','cdd22_3gun','cdd22_7gun','cdd22_14gun',
        'sic_3gun','sic_7gun','hdd18_3gun','hdd18_7gun','gunluk_fark','cdd_max30','sic_max_3gun']

train = train.merge(hava[['yer','tarih'] + HAVA], on=['yer','tarih'], how='left')
test  = test.merge(hava[['yer','tarih'] + HAVA],  on=['yer','tarih'], how='left')

print('eksik sıcaklık - train:', train['sic_ort'].isna().sum(), '| test:', test['sic_ort'].isna().sum())
print()
# Sıcaklık, yilin_gunu'nun aksine eğitim ve test arasında örtüşüyor mu?
print('sic_ort aralığı  train: %.1f – %.1f' % (train['sic_ort'].min(), train['sic_ort'].max()))
print('                 test : %.1f – %.1f' % (test['sic_ort'].min(), test['sic_ort'].max()))
print()
print('Aylık ortalama CDD/HDD ve tüketim (V şekli görünüyor mu):')
print(train.groupby('ay').agg(cdd22=('cdd22','mean'), hdd18=('hdd18','mean'),
                              tuketim=('tuketim_duzeltilmis','median')).round(2).to_string())

## 2026 yazı 2025'ten serin geçmiş

Bu, hava durumunun neden bu kadar işe yaradığını açıklayan asıl nokta. Sadece takvime
bakan bir model "Temmuz = Temmuz" varsayar ve 2026 Temmuz'unu 2025 Temmuz'u gibi tahmin eder.
Oysa gerçekte belirgin bir fark var.

In [ ]:
k = hava.copy()
k['yil'], k['ay'] = k['tarih'].dt.year, k['tarih'].dt.month
karsilastirma = (k[k['ay'].isin([4,5,6,7])]
                 .groupby(['ay','yil'])[['sic_ort','sic_max','cdd22']].mean().round(2))
print(karsilastirma.to_string())

p = karsilastirma.unstack('yil')
print()
print('Nisan-Temmuz toplam CDD22:  2025=%.2f  2026=%.2f  (oran %.2f)' % (
    k[(k['yil']==2025) & (k['ay'].isin([4,5,6,7]))]['cdd22'].mean(),
    k[(k['yil']==2026) & (k['ay'].isin([4,5,6,7]))]['cdd22'].mean(),
    k[(k['yil']==2026) & (k['ay'].isin([4,5,6,7]))]['cdd22'].mean() /
    k[(k['yil']==2025) & (k['ay'].isin([4,5,6,7]))]['cdd22'].mean()))

## Özellik üretimi

Diğer özellikler önceki notebook'larla aynı: shrinkage'lı grup istatistiği, trafo profili
(log uzayında), hafta günü profili. Hepsi `origin_tarihi` öncesindeki veriden hesaplanıyor.

In [ ]:
def trafo_profili(origin):
    g = train[train['tarih'] < origin]
    p = g.groupby('tanim').agg(
        trafo_ortalama=('tuketim_duzeltilmis','mean'), trafo_medyan=('tuketim_duzeltilmis','median'),
        trafo_gun_sayisi=('log_t','count'), log_ortalama=('log_t','mean'),
        log_medyan=('log_t','median'), log_std=('log_t','std'),
        sifir_orani=('tuketim_duzeltilmis', lambda x: (x == 0).mean())).reset_index()
    for w in [30, 90]:
        s = g[g['tarih'] >= origin - pd.Timedelta(days=w)]
        o = s.groupby('tanim').agg(**{
            f'son{w}_log_ort': ('log_t','mean'),
            f'son{w}_sifir': ('tuketim_duzeltilmis', lambda x: (x == 0).mean())}).reset_index()
        p = p.merge(o, on='tanim', how='left')
    p['log_trend'] = p['son30_log_ort'] - p['son90_log_ort']
    return p


def gun_profili(origin):
    g = train[train['tarih'] < origin][['tanim','haftanin_gunu','tuketim_duzeltilmis']].copy()
    o = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_o')
    g = g.merge(o, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_o'].replace(0, np.nan)
    return g.groupby(['tanim','haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


def veri_seti(hedef, origin, k=30):
    g = train[train['tarih'] < origin]
    genel = g['log_t'].mean()
    grup = g.groupby(['il','bolge','guc_bucket'], observed=True)['log_t'].agg(['sum','count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k*genel) / (grup['count'] + k)
    ilguc = g.groupby(['il','guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef.merge(grup[['il','bolge','guc_bucket','grup_log_ort']], on=['il','bolge','guc_bucket'], how='left')
    df = df.merge(ilguc, on=['il','guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel)
    df = df.merge(trafo_profili(origin), on='tanim', how='left')
    df = df.merge(gun_profili(origin), on=['tanim','haftanin_gunu'], how='left')
    return df


def origin_seti(bas, bit, origin):
    return veri_seti(train[(train['tarih'] >= bas) & (train['tarih'] <= bit)], pd.Timestamp(origin))


TEMEL = ['guc','ay','gun','haftanin_gunu','hafta_sonu','yilin_gunu','tatil','il','bolge','ilce',
         'guc_bucket_str','grup_log_ort','trafo_ortalama','trafo_medyan','trafo_gun_sayisi',
         'log_ortalama','log_medyan','log_std','sifir_orani','son30_log_ort','son90_log_ort',
         'son30_sifir','son90_sifir','log_trend','trafo_gun_orani']
OZELLIKLER = TEMEL + HAVA

def rmsle(gercek, tahmin):
    return np.sqrt(np.mean((np.log1p(tahmin) - np.log1p(gercek))**2))

## İki ayrı doğrulama seti

Şimdiye kadarki doğrulamamız kış dönemindeydi (Aralık-Mart) ve orada CDD sıfır — yani
sıcaklığın asıl değerini ölçemiyordu. Bu yüzden **yaz üzerinde çalışan** ikinci bir
doğrulama kurduk:

| | Eğitim origin'leri | Doğrulama dönemi | Ufuk |
|---|---|---|---|
| Yaz | Mar/Nis/May 2025 | Ağu–Kas 2025 | 122 gün |
| Kış | Nis/May/Haz + Ağu/Eki 2025 | Ara 2025 – Mar 2026 | 121 gün |

İkisinde de ufuk gerçek testle (122 gün) aynı. Yaz seti test mevsimine yakın olduğu için
asıl referansımız o, ama bir değişikliğin ikisinde birden iyileştirme yapması gerekiyor —
tek bir sette iyileşme rastlantı olabilir.

In [ ]:
yaz_eg = pd.concat([origin_seti(o, '2025-07-31', o) for o in
                    ['2025-03-01','2025-04-01','2025-05-01']], ignore_index=True)
yaz_va = origin_seti('2025-08-01', '2025-11-30', '2025-08-01')

kis_eg = pd.concat([origin_seti(o, '2025-09-30', o) for o in ['2025-04-01','2025-05-01','2025-06-01']] +
                   [origin_seti(o, '2025-11-30', o) for o in ['2025-08-01','2025-10-01']], ignore_index=True)
kis_va = origin_seti('2025-12-01', '2026-03-31', '2025-12-01')

for d in [yaz_eg, yaz_va, kis_eg, kis_va]:
    for col in KATEGORIK:
        d[col] = d[col].astype('category')
    d['hedef_log'] = np.log1p(d['tuketim_duzeltilmis'])

print('yaz  eğitim %s  doğrulama %s' % (yaz_eg.shape, yaz_va.shape))
print('kış  eğitim %s  doğrulama %s' % (kis_eg.shape, kis_va.shape))

In [ ]:
def olc(E, V, ozellikler, **kw):
    p = dict(n_estimators=3000, learning_rate=0.05, num_leaves=63, random_state=42, verbose=-1)
    p.update(kw)
    m = lgb.LGBMRegressor(**p)
    m.fit(E[ozellikler], E['hedef_log'],
          eval_set=[(V[ozellikler], V['hedef_log'])], eval_metric='rmse',
          categorical_feature=KATEGORIK, callbacks=[lgb.early_stopping(150, verbose=False)])
    t = np.clip(np.expm1(m.predict(V[ozellikler])), 0, None)
    return rmsle(V['tuketim_duzeltilmis'], t), m.best_iteration_


def karsilastir(etiket, ozellikler, **kw):
    a, na = olc(yaz_eg, yaz_va, ozellikler, **kw)
    b, nb = olc(kis_eg, kis_va, ozellikler, **kw)
    print(f'{etiket:38s} yaz={a:.4f}(n={na:<4}) kış={b:.4f}(n={nb:<4}) ort={(a+b)/2:.4f}')


karsilastir('hava durumu yok', TEMEL)
karsilastir('sadece sic_ort', TEMEL + ['sic_ort'])
karsilastir('cdd22 + hdd18', TEMEL + ['cdd22','hdd18'])
karsilastir('+ termal atalet', TEMEL + ['cdd22','hdd18','cdd22_3gun','cdd22_7gun','sic_3gun','sic_7gun'])
karsilastir('tam paket (15 özellik)', OZELLIKLER)

## Hiperparametre: aşırı öğrenmeyi engellemek

122 günlük ufukta model çok hızlı ezberliyor. Varsayılan ayarla (63 yaprak, lr 0.05)
optimum ~55 ağaçta; 150 ağaçta skor belirgin şekilde kötüleşiyor. Bu, önceki
submission'da kullandığımız ayarın optimumun epey ötesinde olduğu anlamına geliyordu.

Çözüm daha basit ağaçlar: az yaprak + düşük öğrenme oranı + her ağaçta özelliklerin
bir kısmını kullanmak. Basit ağaçlar trafoya özel örüntüleri ezberleyemiyor, 122 gün
sonrasında da geçerli kalan genel ilişkileri öğreniyor.

In [ ]:
karsilastir('varsayılan (63 yaprak, lr .05)', OZELLIKLER)
karsilastir('31 yaprak', OZELLIKLER, num_leaves=31)
karsilastir('15 yaprak + lr .03', OZELLIKLER, num_leaves=15, learning_rate=0.03)
karsilastir('15 yaprak + lr .03 + ff .7', OZELLIKLER, num_leaves=15, learning_rate=0.03, colsample_bytree=0.7)

## Ağaç sayısı

Doğrulamalar farklı büyüklükteki eğitim setleriyle yapıldı, final model ise daha büyük bir
setle eğitilecek. Optimum ağaç sayısının veri miktarıyla kayıp kaymadığını kontrol ettik:
826 bin / 1.37 milyon / 1.88 milyon satırlık eğitim setlerinde optimum sırasıyla 120 / 100 / 120
çıktı — yani sabit. Final için 120 kullanıyoruz.

In [ ]:
AYAR = dict(num_leaves=15, learning_rate=0.03, colsample_bytree=0.7)
AGAC = 120

m = lgb.LGBMRegressor(n_estimators=600, random_state=42, verbose=-1, **AYAR)
m.fit(yaz_eg[OZELLIKLER], yaz_eg['hedef_log'], categorical_feature=KATEGORIK)
for n in [80, 100, 120, 150, 200, 300, 450, 600]:
    t = np.clip(np.expm1(m.predict(yaz_va[OZELLIKLER], num_iteration=n)), 0, None)
    print(f'  yaz  n={n:<4} {rmsle(yaz_va["tuketim_duzeltilmis"], t):.4f}')

## Final model ve submission

Eğitim origin'leri iki gruptan: yaz hedefleri (Nisan-Eylül 2025) ve kış hedefleri
(Aralık 2025 - Mart 2026). Yaz grubu modelin sıcak hava davranışını görmesini,
kış grubu ise zengin profillerle (11-13 aylık geçmiş) çalışmasını sağlıyor.

Tahmin origin'i 1 Nisan 2026 — yani train'in tamamı profil hesabında kullanılabiliyor.
Üç farklı tohumla eğitip log uzayında ortalama alıyoruz; bedava küçük bir kararlılık kazancı.

In [ ]:
YAZ_ORIGIN = ['2025-04-01','2025-05-01','2025-06-01']
KIS_ORIGIN = ['2025-12-01','2026-01-01','2026-02-01']

egitim = pd.concat([origin_seti(o, '2025-09-30', o) for o in YAZ_ORIGIN] +
                   [origin_seti(o, '2026-03-31', o) for o in KIS_ORIGIN], ignore_index=True)
for col in KATEGORIK:
    egitim[col] = egitim[col].astype('category')
egitim['hedef_log'] = np.log1p(egitim['tuketim_duzeltilmis'])

test_seti = veri_seti(test, pd.Timestamp('2026-04-01'))
# Kategori sıralaması eğitimle birebir aynı olmalı; LightGBM kategorileri kod numarasıyla
# kullandığı için farklı sıralama sessizce yanlış tahmin üretir
for col in KATEGORIK:
    test_seti[col] = pd.Categorical(test_seti[col], categories=egitim[col].cat.categories)

print('eğitim', egitim.shape, '| test', test_seti.shape)
print('kategori uyumu:', all(list(test_seti[c].cat.categories) == list(egitim[c].cat.categories)
                             for c in KATEGORIK))
print('test\'te eksik hava verisi:', int(test_seti[HAVA].isna().sum().sum()))

In [ ]:
log_tahminler = []
for tohum in [42, 7, 2024]:
    m = lgb.LGBMRegressor(n_estimators=AGAC, random_state=tohum, verbose=-1, **AYAR)
    m.fit(egitim[OZELLIKLER], egitim['hedef_log'], categorical_feature=KATEGORIK)
    log_tahminler.append(m.predict(test_seti[OZELLIKLER]))

tahmin = np.clip(np.expm1(np.mean(log_tahminler, axis=0)), 0, None)
submission = pd.DataFrame({'id': test_seti['id'], 'tuketim': tahmin})

ornek = pd.read_csv(RAW + 'sample_submission.csv')
print('satır sayısı eşit:', len(submission) == len(ornek))
print('id kümeleri aynı :', set(submission['id']) == set(ornek['id']))
print('NaN / negatif    :', submission['tuketim'].isna().sum(), '/', (submission['tuketim'] < 0).sum())
print()
print(submission['tuketim'].describe().round(1).to_string())

submission.to_csv('../submissions/sub_03_hava_durumu.csv', index=False)
print('\nkaydedildi: submissions/sub_03_hava_durumu.csv')

In [ ]:
# Önceki submission ile kıyas: yaz rampası neden düştü?
onceki = pd.read_csv('../submissions/sub_02_yaz_dahil.csv')
kiyas = onceki.merge(submission, on='id', suffixes=('_onceki','_yeni'))
kiyas['ay'] = pd.to_datetime(kiyas['id'].str.split('_').str[-1]).dt.month
print(kiyas.groupby('ay')[['tuketim_onceki','tuketim_yeni']].mean().round(0).to_string())

## Sonuç

| Adım | Yaz doğrulama | Kış doğrulama |
|---|---|---|
| Hava durumu yok | 1.1724 | 1.0467 |
| Sıcaklık paketi (15 özellik) | 1.0495 | 1.0285 |
| + hiperparametre ayarı | **1.0086** | **1.0030** |

Yaz doğrulamasında toplam iyileşme 0.164. Üç farklı tohumla tekrarlandığında sonuç
kararlı çıktı (1.0495 / 1.0536 / 1.0515).

**Denenip elenenler:**

- *Trafo bazlı sıcaklık hassasiyeti* (her trafonun CDD'ye tepki eğimi): eğitim setinde
  sadece %7 dolabildiği için zarar verdi (1.0914 → 1.1144). Erken origin'lerin geçmişinde
  yaz olmadığından eğim hesaplanamıyor.
- *Hissedilen sıcaklık, güneş radyasyonu, güneşlenme süresi, buharlaşma (sulama talebi
  göstergesi), yağış, rüzgâr*: hiçbiri katkı sağlamadı, hepsi sıcaklık paketinden biraz
  daha kötü. Sıcaklık + derece-gün + termal atalet hava sinyalini zaten tüketiyor.
- *CDD eşiği*: 18/20/22/24/26 denendi, 22°C en iyisi.

**Yeni tahminler neden daha düşük:** 2026 yazı 2025'ten belirgin şekilde serin geçti
(Temmuz ortalaması 27.5°C vs 30.0°C, soğutma yükü %31 daha az). Takvime bakan önceki model
"Temmuz = Temmuz" varsaydığı için yüksek tahmin ediyordu. Yeni model gerçekleşen havayı
gördüğü için aşağı çekiyor — düşük tahmin bir kalibrasyon hatası değil, doğru bilgiye
verilen tepki.